# Student Performance - Regressão
Tarefa de regressão para student performance

# EDA


In [121]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [ ]:
train_dataset = pd.read_csv("../../regressao/train.csv")
test_dataset = pd.read_csv("../../regressao/test.csv")

In [123]:
print(f"Formato do dataset:{train_dataset.head()}")

Formato do dataset:  school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1  at_home     other  ...   
2     GP   F   15       U     LE3       T     1     1  at_home     other  ...   
3     GP   F   15       U     GT3       T     4     2   health  services  ...   
4     GP   F   16       U     GT3       T     3     3    other     other  ...   

  internet romantic  famrel  freetime  goout Dalc Walc health absences  score  
0       no       no       4         3      4    1    1      3        6   5.67  
1      yes       no       5         3      3    1    1      3        4   5.33  
2      yes       no       4         3      2    2    3      3       10   8.33  
3      yes      yes       3         2      2    1    1      5        2  14.67  
4       no       no       4         3      2    1    2      5        4   8.67  

[5 rows x 31 

Lendo o texto, como a característica dos dados é que são muito categóricos, e até os dados ordinais, não parecem que vão ter uma boa relação linear com a nota resultante. Além disso, é visível a dependência entre as variáveis.

# Limpezas e pré processamento



Antes de começar a limpeza, vale separar o que é a função objetivo e o que são as variáveis. Vou aplicando simultâneamente aos dados de teste, dado que eles também devem ser limpos e pré processados

In [124]:
X = train_dataset.drop(columns=['score'])
Y = train_dataset.loc[:, 'score']

A primeira ação a ser feita para limpar os dados, é transformar os dados binários que não são codificados em 0 e 1, para assumirem um desses valores ("yes" e "no" por exemplo)

In [125]:
binary_mappings = {
    'school': {'MS': 0, 'GP': 1},
    'sex': {'F': 0, 'M': 1},
    'address': {'R': 0, 'U': 1},
    'famsize': {'LE3': 0, 'GT3': 1},
    'Pstatus': {'A': 0, 'T': 1},
    'schoolsup': {'no': 0, 'yes': 1},
    'famsup': {'no': 0, 'yes': 1},
    'paid': {'no': 0, 'yes': 1},
    'activities': {'no': 0, 'yes': 1},
    'nursery': {'no': 0, 'yes': 1},
    'higher': {'no': 0, 'yes': 1},
    'internet': {'no': 0, 'yes': 1},
    'romantic': {'no': 0, 'yes': 1}
}


In [126]:
X = X.replace(binary_mappings)
test_dataset = test_dataset.replace(binary_mappings)

Algo que também ficou claro durante a EDA é a presença de dados nominais, que devemos tratar para lidar numericamente

In [127]:
print(f"A mãe pode assumir: {train_dataset['Mjob'].unique()[:]} trabalhos diferentes")

print(f"O pai pode assumir pode assumir: {train_dataset['Fjob'].unique()[:]} trabalhos diferentes")

print(f"As razões de escolha da dada escola podem ser: {train_dataset['reason'].unique()[:]}")

print(f"O guardião legal do aluno pode ser: {train_dataset['guardian'].unique()[:]}")

A mãe pode assumir: <ArrowStringArray>
['at_home', 'health', 'other', 'services', 'teacher']
Length: 5, dtype: str trabalhos diferentes
O pai pode assumir pode assumir: <ArrowStringArray>
['teacher', 'other', 'services', 'health', 'at_home']
Length: 5, dtype: str trabalhos diferentes
As razões de escolha da dada escola podem ser: <ArrowStringArray>
['course', 'other', 'home', 'reputation']
Length: 4, dtype: str
O guardião legal do aluno pode ser: <ArrowStringArray>
['mother', 'father', 'other']
Length: 3, dtype: str


Para tornar os dados numéricos comparáveis, a ideia é usar **One Hot Encoding**, cada nome de cada variável nominal vai se tornar uma variável binária, e assim lidamos bem com as variáveis nominais. 

Porém, verifiquei que isso é um problema para os problemas de regressão, pois torna a matriz de variáveis LD, e leva o sistema à ter infinitas soluções. Por isso, criei duas base de dados diferentes: uma que vai para a regressão, e não usa os campos nominais, e outra que vai para random forest/decision tree, que usa os campos nominais.

Campos nominais: Mjob Fjob, reason, guardian

In [128]:
nominal_cols = ['Mjob', 'Fjob', 'reason', 'guardian']
#Cópias, para criar as duas databases diferentes
X_tree = X.copy()
X_regr = X.copy()

#Os datasets de teste, que vão dar o resultado depois
test_dataset_tree = test_dataset.copy()
test_dataset_regr = test_dataset.copy()
#Agora, percorrendo os campos nominais
for col in nominal_cols:
    # pegar todas as novas variaveis à ser criadas
    categorias_unicas = X[col].unique()
    
    categorias_codificar = categorias_unicas
    #Para cada categoria, elimino as colunas, e, no caso da base para decision trees
    for category in categorias_codificar:
        new_col_name = f"{col}_{category}"
        X_tree[new_col_name] = (X[col] == category).astype(int)
        test_dataset_tree[new_col_name] =  (test_dataset[col] == category).astype(int)
        
    X_tree = X_tree.drop(columns=[col])
    X_regr = X_regr.drop(columns=[col])

    test_dataset_tree = test_dataset_tree.drop(columns=[col])
    test_dataset_regr = test_dataset_regr.drop(columns=[col])
print(X_tree.head())
print(X_regr)

  school sex  age address famsize Pstatus  Medu  Fedu  traveltime  studytime  \
0      1   0   18       1       1       0     4     4           2          2   
1      1   0   17       1       1       1     1     1           1          2   
2      1   0   15       1       0       1     1     1           1          2   
3      1   0   15       1       1       1     4     2           1          3   
4      1   0   16       1       1       1     3     3           1          2   

   ...  Fjob_services Fjob_health Fjob_at_home reason_course reason_other  \
0  ...              0           0            0             1            0   
1  ...              0           0            0             1            0   
2  ...              0           0            0             0            1   
3  ...              1           0            0             0            0   
4  ...              0           0            0             0            0   

  reason_home reason_reputation guardian_mother guardian

Agora, a parte de treinamento

In [ ]:
model_lr = LinearRegression()
model_lr.fit(X_regr, Y)



#Em ambos, evitar overfitting com max_depth: não permite que analise todos os nós
model_dt = DecisionTreeRegressor(criterion='squared_error',max_depth=6)
model_dt.fit(X=X_tree, y=Y)



model_rf = RandomForestRegressor(n_estimators=20,criterion='squared_error',max_depth=8)
model_rf.fit(X=X_tree,y=Y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",20
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number 

Agora, testando para ver a taxa de underfitting.

In [130]:

pred_novo_lr = model_lr.predict(X_regr)
pred_novo_dt = model_dt.predict(X_tree)
pred_novo_rf = model_rf.predict(X_tree)


resultados_df = pd.DataFrame({
    'Predicao_LinearRegression': pred_novo_lr,
    'Predicao_DecisionTree': pred_novo_dt,
    'Predicao_RandomForest': pred_novo_rf,
    'score':Y

})

In [131]:
resultados_df["score linearregression"] = (resultados_df['Predicao_LinearRegression']-resultados_df['score'])**2
resultados_df["score decisiontree"] = (resultados_df['Predicao_DecisionTree']-resultados_df['score'])**2
resultados_df["score randomforest"] = (resultados_df['Predicao_RandomForest']-resultados_df['score'])**2
print(np.sqrt(resultados_df["score linearregression"].mean()))
print(np.sqrt(resultados_df["score decisiontree"].mean()))
print(np.sqrt(resultados_df["score randomforest"].mean()))

3.1702731440269893
2.54020805692098
1.5659125319183635


O ideal é ter os valores controlados, para evitar overfitting. Por isso, vou escolher o do meio: nem teve um overfitting tão alto, nem underfitting tão baixo, parece ter aprendido melhor. Com os modelos treinados, vamos prever os valores do test dataset.

In [132]:
prediction_result_rf = model_rf.predict(test_dataset_tree)
resultado_final_rf = pd.DataFrame({
    'id': range(len(prediction_result_rf)),
    'score': prediction_result_rf
})
resultado_final_rf.to_csv("submission.csv", index=False)

prediction_result_dt = model_dt.predict(test_dataset_tree)
resultado_final_dt = pd.DataFrame({
    'id': range(len(prediction_result_dt)),
    'score': prediction_result_dt
})

resultado_final_dt.to_csv("submission_dt.csv", index=False)

prediction_result_lr = model_lr.predict(test_dataset_regr)
resultado_final_lr = pd.DataFrame({
    'id': range(len(prediction_result_lr)),
    'score': prediction_result_lr
})
resultado_final_lr.to_csv("submission_lr.csv", index=False)


Resultados: Usando LR sem as variáveis nominais, DT com 13 max depth, e RF com 15 max depth, o LR teve melhor resultado. Isso indica claramente um alto overfitting dos outros dois modelos. Portanto, após esse resultado, usei novamente DT e RF, mas com 6 e 8 de max depth, respectivamente.

O resultado disso foi: DT melhorou muito (overfitting), e RF piorou (possivelmente underfitting).

O fato do LR ser melhor é ou devido à falta de fine tuning dos outros modelos, nesse caso, ou devido ao fato de as variáveis nominais que usei para treinar o DT e o RF, mas não usei para o LR, poderem ser dependentes de outras variáveis e não acrescentarem tanta informação, apesar de contribuirem no corte.